# Pixel Lab — bản trình diễn trên Google Colab

Notebook này dành cho giảng viên chạy thử dự án mà không cần sửa mã nguồn.

## Ba bước để chạy demo

1. Chọn **Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU**, sau đó bấm **Kết nối**.
2. Mở biểu tượng **chìa khóa** bên trái, tạo secret `NGROK_AUTHTOKEN` từ [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) và bật quyền truy cập. Chế độ xử lý thủ công không cần khóa OpenAI.
3. Chạy lần lượt ba cell được đánh số **1 → 2 → 3**, rồi bấm nút xanh **MỞ GIAO DIỆN PIXEL LAB**.

> Cell số 2 tự tải đủ 300 ảnh BSDS300, build C++/OpenMP/CUDA và chạy kiểm thử. Khi thấy **CÀI ĐẶT HOÀN TẤT**, chuyển sang cell số 3. Khi Colab ngắt runtime, chạy lại từ bước 1.

In [ ]:
#@title 1. Kiểm tra GPU
print('Đang kiểm tra GPU và CUDA...')
!nvidia-smi
!nvcc --version
print('GPU đã sẵn sàng.')

In [ ]:
#@title 2. Cài đặt dự án và chạy kiểm thử (chỉ chạy một lần)
REPOSITORY = 'https://github.com/Chicken20145/ai-assisted-parallel-image-processing.git'
GIT_REF = 'main'

%cd /content
!test -d ai-assisted-parallel-image-processing || git clone --branch {GIT_REF} {REPOSITORY}
%cd /content/ai-assisted-parallel-image-processing
!git fetch origin {GIT_REF}
!git switch {GIT_REF}
!git pull --ff-only origin {GIT_REF}

import subprocess
from pathlib import Path

def show_git_sync():
    local_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    remote_commit = subprocess.check_output(['git', 'rev-parse', f'origin/{GIT_REF}'], text=True).strip()
    if local_commit != remote_commit:
        raise RuntimeError(f'Colab chưa đồng bộ GitHub: local={local_commit[:7]}, GitHub={remote_commit[:7]}')
    print(f'ĐÃ ĐỒNG BỘ GITHUB — branch: {GIT_REF} — commit: {local_commit[:7]}')

show_git_sync()
!bash scripts/setup_colab.sh

dataset_dir = Path('/content/ai-assisted-parallel-image-processing/data/external/BSDS300/images')
image_count = sum(1 for path in dataset_dir.rglob('*') if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
core_cli = Path('/content/ai-assisted-parallel-image-processing/build-colab/image_pipeline_cli')
if image_count != 300 or not core_cli.is_file():
    raise RuntimeError(f'Thiết lập chưa hoàn tất: tìm thấy {image_count}/300 ảnh, core={core_cli.is_file()}')
print('\n' + '=' * 60)
print('CÀI ĐẶT HOÀN TẤT')
print(f'Dataset BSDS300: {image_count}/300 ảnh')
print('Core C++/OpenMP/CUDA: đã build và kiểm thử')
print('Tiếp theo: chạy cell số 3 để mở giao diện.')
print('=' * 60)

## Làm mới phiên bản từ GitHub (tùy chọn)

Không cần chạy mục này trong lần demo đầu tiên. Chỉ chạy cell dưới khi nhóm vừa cập nhật nhánh `main` và muốn lấy mã nguồn mới mà không khởi động lại Colab.

In [ ]:
#@title Cập nhật mã nguồn và kiểm thử lại (tùy chọn)
%cd /content/ai-assisted-parallel-image-processing
!git fetch origin {GIT_REF}
!git switch {GIT_REF}
!git pull --ff-only origin {GIT_REF}
show_git_sync()
!bash scripts/build_colab.sh
!bash scripts/test_colab.sh

## Demo tương tác

Chạy cell số 3 và bấm nút xanh để mở giao diện. Trong giao diện, giảng viên có thể:

- **Xử lý một ảnh:** chọn ảnh, thuật toán và cách chạy rồi xem ảnh kết quả cùng thời gian thực thi.
- **Pipeline nhiều bước:** ghép Gaussian Blur, Sobel và Cân bằng histogram.
- **Benchmark 300 ảnh:** bấm một nút để chạy ma trận thuật toán/backend và tải CSV kết quả.
- **Độ chính xác:** xem tổng sai lệch, MAE và MSE dưới dạng phân số chính xác, không làm tròn giả.

Giữ cell số 3 và runtime Colab đang chạy trong suốt buổi demo. URL chỉ tồn tại trong phiên hiện tại.

In [ ]:
#@title 3. Mở giao diện tương tác
import getpass
import os
import subprocess
import sys
import time
from pathlib import Path

import requests
from IPython.display import HTML, display

try:
    from pyngrok import ngrok
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'])
    from pyngrok import ngrok

repo = Path('/content/ai-assisted-parallel-image-processing')
core_cli = repo / 'build-colab' / 'image_pipeline_cli'
if not core_cli.is_file():
    raise FileNotFoundError('Chưa có image_pipeline_cli. Hãy chạy cell setup/build trước.')

if '_pixel_lab_process' in globals() and _pixel_lab_process.poll() is None:
    _pixel_lab_process.terminate()
    try:
        _pixel_lab_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        _pixel_lab_process.kill()
if '_pixel_lab_log' in globals() and not _pixel_lab_log.closed:
    _pixel_lab_log.close()

environment = os.environ.copy()
environment['PIP_CORE_CLI'] = str(core_cli)
log_path = Path('/tmp/pixel_lab_streamlit.log')
_pixel_lab_log = log_path.open('w', encoding='utf-8')
_pixel_lab_process = subprocess.Popen(
    [sys.executable, '-m', 'streamlit', 'run', 'app/app.py',
     '--server.address=0.0.0.0', '--server.port=8501',
     '--server.headless=true', '--server.enableCORS=false',
     '--server.enableXsrfProtection=false'],
    cwd=repo, env=environment, stdout=_pixel_lab_log,
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    if _pixel_lab_process.poll() is not None:
        _pixel_lab_log.flush()
        raise RuntimeError(log_path.read_text(encoding='utf-8', errors='replace'))
    try:
        if requests.get('http://127.0.0.1:8501/_stcore/health', timeout=1).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    _pixel_lab_process.terminate()
    _pixel_lab_log.flush()
    raise TimeoutError(log_path.read_text(encoding='utf-8', errors='replace'))

try:
    from google.colab import userdata
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
except Exception:
    ngrok_token = getpass.getpass('Nhập ngrok authtoken (ký tự sẽ được ẩn): ')
if not ngrok_token:
    raise ValueError('Thiếu NGROK_AUTHTOKEN. Hãy thêm token trong Colab Secrets rồi chạy lại cell.')

ngrok.set_auth_token(ngrok_token)
del ngrok_token
ngrok.kill()
_pixel_lab_tunnel = ngrok.connect(8501, bind_tls=True)
ui_url = _pixel_lab_tunnel.public_url
display(HTML(f'''
<div style="padding:24px;border:1px solid #d0d7de;border-radius:12px;background:#f6f8fa;max-width:760px">
  <h2 style="margin:0 0 8px;color:#1f2328">Giao diện đã sẵn sàng</h2>
  <p style="margin:0 0 18px;color:#57606a">Bấm nút dưới để mở Pixel Lab trong tab mới.</p>
  <a href="{ui_url}" target="_blank" style="display:inline-block;padding:14px 24px;border-radius:8px;background:#0969da;color:white;text-decoration:none;font-size:18px;font-weight:700">MỞ GIAO DIỆN PIXEL LAB</a>
  <p style="margin:16px 0 0;color:#57606a;font-size:13px">Giữ runtime và cell này hoạt động trong khi demo.</p>
</div>
'''))
print('Server đang chạy. Log:', log_path)
print('Khi dùng xong, chạy ngrok.disconnect(ui_url) để đóng tunnel.')

## Lưu kết quả lên Google Drive (tùy chọn)

Chỉ mount Drive khi cần lưu kết quả. Nên benchmark trên ổ đĩa cục bộ `/content` rồi mới chép CSV/biểu đồ sang Drive để I/O mạng không làm sai lệch thời gian.

In [ ]:
#@title 4. Sao chép CSV và biểu đồ sang Google Drive (tùy chọn)
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil

source = Path('/content/ai-assisted-parallel-image-processing/benchmarks/results')
destination = Path('/content/drive/MyDrive/parallel-image-processing/results')
if source.exists():
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, destination, dirs_exist_ok=True)
    print(f'Copied results to {destination}')
else:
    print('No benchmark results yet.')